# Decision Tree (ID3) - Tự cài đặt từ đầu, không dùng thư viện

Bài toán kinh điển: **Play Tennis** — dựa vào thời tiết, dự đoán có nên chơi tennis không (Yes/No).

Notebook này tự code toàn bộ thuật toán **ID3**: Entropy → Information Gain → xây cây đệ quy → dự đoán.

## 1. Import thư viện cần thiết

Chỉ dùng thư viện chuẩn của Python (`math` để tính log, `collections.Counter` để đếm), **không dùng sklearn hay bất kỳ thư viện ML nào**.

In [1]:
import math
from collections import Counter

## 2. Dữ liệu: 14 ngày thời tiết

| Thuộc tính | Các giá trị có thể |
|---|---|
| Outlook | Sunny, Overcast, Rain |
| Temperature | Hot, Mild, Cool |
| Humidity | High, Normal |
| Wind | Weak, Strong |
| **Play (nhãn)** | Yes, No |

In [2]:
dataset = [
    #  Outlook,   Temperature, Humidity, Wind,    Play
    ("Sunny",    "Hot",  "High",   "Weak",   "No"),
    ("Sunny",    "Hot",  "High",   "Strong", "No"),
    ("Overcast", "Hot",  "High",   "Weak",   "Yes"),
    ("Rain",     "Mild", "High",   "Weak",   "Yes"),
    ("Rain",     "Cool", "Normal", "Weak",   "Yes"),
    ("Rain",     "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"),
    ("Sunny",    "Mild", "High",   "Weak",   "No"),
    ("Sunny",    "Cool", "Normal", "Weak",   "Yes"),
    ("Rain",     "Mild", "Normal", "Weak",   "Yes"),
    ("Sunny",    "Mild", "Normal", "Strong", "Yes"),
    ("Overcast", "Mild", "High",   "Strong", "Yes"),
    ("Overcast", "Hot",  "Normal", "Weak",   "Yes"),
    ("Rain",     "Mild", "High",   "Strong", "No"),
]

attribute_names = ["Outlook", "Temperature", "Humidity", "Wind"]

# In bảng dữ liệu cho dễ nhìn
print(f"{'Outlook':<10}{'Temp':<7}{'Humidity':<10}{'Wind':<8}{'Play'}")
for row in dataset:
    print(f"{row[0]:<10}{row[1]:<7}{row[2]:<10}{row[3]:<8}{row[4]}")

Outlook   Temp   Humidity  Wind    Play
Sunny     Hot    High      Weak    No
Sunny     Hot    High      Strong  No
Overcast  Hot    High      Weak    Yes
Rain      Mild   High      Weak    Yes
Rain      Cool   Normal    Weak    Yes
Rain      Cool   Normal    Strong  No
Overcast  Cool   Normal    Strong  Yes
Sunny     Mild   High      Weak    No
Sunny     Cool   Normal    Weak    Yes
Rain      Mild   Normal    Weak    Yes
Sunny     Mild   Normal    Strong  Yes
Overcast  Mild   High      Strong  Yes
Overcast  Hot    Normal    Weak    Yes
Rain      Mild   High      Strong  No


## 3. Chuyển dữ liệu sang dạng `(features_dict, label)`

Dễ thao tác hơn khi xây cây — mỗi mẫu là 1 tuple gồm dict thuộc tính và nhãn.

In [3]:
data = []
for row in dataset:
    features = {
        "Outlook": row[0],
        "Temperature": row[1],
        "Humidity": row[2],
        "Wind": row[3],
    }
    label = row[4]
    data.append((features, label))

# Xem thử 1 mẫu
print(data[0])

({'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak'}, 'No')


## 4. Hàm tính Entropy

Entropy đo độ "lẫn lộn" của nhãn trong 1 tập dữ liệu:

$$Entropy(S) = -\sum_i p_i \cdot \log_2(p_i)$$

- Entropy = 0 → tất cả cùng 1 nhãn (rất "thuần khiết")
- Entropy = 1 → 2 nhãn chia đều 50/50 (rất "lẫn lộn")

In [4]:
def entropy(samples):
    total = len(samples)
    if total == 0:
        return 0

    # đếm số lượng từng nhãn (Yes/No)
    count_yes_no = Counter(label for _, label in samples)

    result = 0
    for label, count in count_yes_no.items():
        p = count / total                  # tỉ lệ của nhãn này
        result -= p * math.log2(p)         # công thức Entropy
    return result

### Thử tính Entropy của toàn bộ tập dữ liệu ban đầu

In [5]:
e = entropy(data)
count = Counter(label for _, label in data)
print(f"Số lượng Yes/No: {dict(count)}")
print(f"Entropy(S) = {e:.4f}")

Số lượng Yes/No: {'No': 5, 'Yes': 9}
Entropy(S) = 0.9403


## 5. Hàm chia dữ liệu theo 1 thuộc tính

Ví dụ chia theo `Outlook` sẽ ra 3 nhóm: `Sunny`, `Overcast`, `Rain`.

In [6]:
def split_data(samples, attribute):
    groups = {}  # vd: {"Sunny": [...], "Overcast": [...], "Rain": [...]}
    for features, label in samples:
        value = features[attribute]           # vd: "Sunny"
        if value not in groups:
            groups[value] = []
        groups[value].append((features, label))
    return groups

### Thử chia dữ liệu theo `Outlook`

In [7]:
groups = split_data(data, "Outlook")
for value, group_samples in groups.items():
    labels_in_group = [label for _, label in group_samples]
    print(f"Outlook = {value}: {len(group_samples)} mẫu, nhãn = {labels_in_group}")

Outlook = Sunny: 5 mẫu, nhãn = ['No', 'No', 'No', 'Yes', 'Yes']
Outlook = Overcast: 4 mẫu, nhãn = ['Yes', 'Yes', 'Yes', 'Yes']
Outlook = Rain: 5 mẫu, nhãn = ['Yes', 'Yes', 'No', 'Yes', 'No']


## 6. Hàm tính Information Gain

Information Gain đo mức độ "làm sạch" dữ liệu khi chia theo 1 thuộc tính:

$$Gain(S, A) = Entropy(S) - \sum_{v} \frac{|S_v|}{|S|} \cdot Entropy(S_v)$$

Gain càng lớn → thuộc tính đó càng giúp phân loại tốt.

In [8]:
def information_gain(samples, attribute):
    entropy_before = entropy(samples)

    groups = split_data(samples, attribute)
    total = len(samples)

    entropy_after = 0
    for value, group_samples in groups.items():
        weight = len(group_samples) / total       # tỉ trọng của nhóm này
        entropy_after += weight * entropy(group_samples)

    return entropy_before - entropy_after

### Tính Information Gain của từng thuộc tính (ở gốc cây)

In [9]:
for attr in attribute_names:
    gain = information_gain(data, attr)
    print(f"Gain({attr:<12}) = {gain:.4f}")

Gain(Outlook     ) = 0.2467
Gain(Temperature ) = 0.0292
Gain(Humidity    ) = 0.1518
Gain(Wind        ) = 0.0481


## 7. Hàm chọn thuộc tính tốt nhất (Gain lớn nhất) để chia

In [10]:
def best_attribute(samples, remaining_attributes):
    best_attr = None
    best_gain = -1

    for attr in remaining_attributes:
        gain = information_gain(samples, attr)
        if gain > best_gain:
            best_gain = gain
            best_attr = attr

    return best_attr, best_gain

### Thử tìm thuộc tính tốt nhất ở gốc

In [11]:
best_attr, best_gain = best_attribute(data, attribute_names)
print(f"Thuộc tính tốt nhất để chia ở gốc: '{best_attr}' (Gain = {best_gain:.4f})")

Thuộc tính tốt nhất để chia ở gốc: 'Outlook' (Gain = 0.2467)


## 8. Cấu trúc 1 node của cây

Mỗi node là:
- **Lá (leaf)**: có `label` (Yes/No) — điểm dừng của cây
- **Node trong**: có `attribute` (thuộc tính dùng để chia) và `children` (các nhánh con)

In [12]:
class Node:
    def __init__(self):
        self.is_leaf = False
        self.label = None          # chỉ dùng khi is_leaf = True
        self.attribute = None      # thuộc tính dùng để chia, vd "Outlook"
        self.children = {}         # vd: {"Sunny": Node(...), "Rain": Node(...)}


def majority_label(samples):
    """Trả về nhãn xuất hiện nhiều nhất trong tập mẫu"""
    labels = [label for _, label in samples]
    return Counter(labels).most_common(1)[0][0]

## 9. Thuật toán ID3 — xây cây đệ quy

Các bước:
1. Nếu tất cả mẫu cùng 1 nhãn → tạo lá
2. Nếu hết thuộc tính để chia → tạo lá, lấy nhãn đa số
3. Ngược lại: chọn thuộc tính tốt nhất, chia dữ liệu, rồi **đệ quy** xây từng nhánh con

In [13]:
def build_tree(samples, remaining_attributes):
    labels = [label for _, label in samples]

    # Trường hợp dừng 1: tất cả mẫu cùng 1 nhãn -> tạo lá (Leaf)
    if len(set(labels)) == 1:
        leaf = Node()
        leaf.is_leaf = True
        leaf.label = labels[0]
        return leaf

    # Trường hợp dừng 2: hết thuộc tính để chia -> tạo lá, lấy nhãn đa số
    if len(remaining_attributes) == 0:
        leaf = Node()
        leaf.is_leaf = True
        leaf.label = majority_label(samples)
        return leaf

    # Chọn thuộc tính tốt nhất để chia ở bước này
    attr, gain = best_attribute(samples, remaining_attributes)

    node = Node()
    node.attribute = attr

    # Chia dữ liệu theo thuộc tính vừa chọn
    groups = split_data(samples, attr)

    # Danh sách thuộc tính còn lại cho các nhánh con (bỏ thuộc tính vừa dùng)
    next_attributes = [a for a in remaining_attributes if a != attr]

    # Đệ quy xây từng nhánh con
    for value, group_samples in groups.items():
        node.children[value] = build_tree(group_samples, next_attributes)

    return node

### Xây cây quyết định từ toàn bộ dữ liệu

In [14]:
tree = build_tree(data, attribute_names)
print("Cây đã được xây xong! (lưu trong biến `tree`)")

Cây đã được xây xong! (lưu trong biến `tree`)


## 10. Hàm in cây ra màn hình (dạng text dễ nhìn)

In [15]:
def print_tree(node, indent=""):
    if node.is_leaf:
        print(f"{indent}=> Play: {node.label}")
        return

    for value, child in node.children.items():
        print(f"{indent}[{node.attribute} = {value}]")
        print_tree(child, indent + "    ")

### In cây quyết định hoàn chỉnh

In [16]:
print_tree(tree)

[Outlook = Sunny]
    [Humidity = High]
        => Play: No
    [Humidity = Normal]
        => Play: Yes
[Outlook = Overcast]
    => Play: Yes
[Outlook = Rain]
    [Wind = Weak]
        => Play: Yes
    [Wind = Strong]
        => Play: No


## 11. Hàm dự đoán (predict) cho 1 ngày mới

Đi từ gốc cây xuống, theo nhánh ứng với giá trị thuộc tính của mẫu, đến khi gặp lá.

In [17]:
def predict(node, sample_features):
    if node.is_leaf:
        return node.label

    value = sample_features[node.attribute]
    child = node.children.get(value)

    if child is None:
        # trường hợp gặp giá trị chưa từng thấy khi huấn luyện
        return "Không xác định"

    return predict(child, sample_features)

## 12. Kiểm tra lại độ chính xác trên tập huấn luyện

In [18]:
correct = 0
for row in dataset:
    features = {"Outlook": row[0], "Temperature": row[1], "Humidity": row[2], "Wind": row[3]}
    true_label = row[4]
    pred = predict(tree, features)
    mark = "✓" if pred == true_label else "✗"
    if pred == true_label:
        correct += 1
    print(f"{features} -> Dự đoán: {pred} | Thực tế: {true_label}  {mark}")

print(f"\nĐộ chính xác: {correct}/{len(dataset)} = {correct/len(dataset)*100:.1f}%")

{'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak'} -> Dự đoán: No | Thực tế: No  ✓
{'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Strong'} -> Dự đoán: No | Thực tế: No  ✓
{'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak'} -> Dự đoán: Yes | Thực tế: Yes  ✓
{'Outlook': 'Rain', 'Temperature': 'Mild', 'Humidity': 'High', 'Wind': 'Weak'} -> Dự đoán: Yes | Thực tế: Yes  ✓
{'Outlook': 'Rain', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak'} -> Dự đoán: Yes | Thực tế: Yes  ✓
{'Outlook': 'Rain', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong'} -> Dự đoán: No | Thực tế: No  ✓
{'Outlook': 'Overcast', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong'} -> Dự đoán: Yes | Thực tế: Yes  ✓
{'Outlook': 'Sunny', 'Temperature': 'Mild', 'Humidity': 'High', 'Wind': 'Weak'} -> Dự đoán: No | Thực tế: No  ✓
{'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak'} ->

## 13. Dự đoán cho ngày mới (chưa từng thấy)

In [19]:
new_day = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}
pred = predict(tree, new_day)
print(f"Thời tiết mới: {new_day}")
print(f"=> Dự đoán: {pred}")

Thời tiết mới: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
=> Dự đoán: No
